# Final Operating Point Audit

Ce notebook reprend le script `final_operating_point_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Selectionne les points de fonctionnement et estime la latence pour le live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Audit validation-selected final operating points and latency estimates.
- Commande de reproduction referencee : final operating points.
- Artefacts controles : Validation-selected final operating-point and latency audit exists. (`runs/exp_033_final_operating_points/metrics/operating_point_summary.csv`).
- Run par defaut : `runs/exp_033_final_operating_points`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_operating_point_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score

from ml_pipeline import ROOT, alarm_episodes, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


CANDIDATES = {
    "mean_temporal_conv": ["cnn1d_aug_focal", "cnn_gru_aug_bce", "tcn_aug_bce", "tcn_aug_focal", "tcn_noaug_bce", "tcn_noaug_focal"],
    "mean_tcn_only": ["tcn_aug_bce", "tcn_aug_focal", "tcn_noaug_bce", "tcn_noaug_focal"],
    "mean_all8": [
        "cnn1d_aug_focal",
        "cnn_gru_aug_bce",
        "gru_noaug_bce",
        "lstm_aug_bce",
        "tcn_aug_bce",
        "tcn_aug_focal",
        "tcn_noaug_bce",
        "tcn_noaug_focal",
    ],
    "tcn_aug_bce": ["tcn_aug_bce"],
    "tcn_aug_focal": ["tcn_aug_focal"],
    "cnn1d_aug_focal": ["cnn1d_aug_focal"],
}


OPERATING_POLICIES = {
    "max_hit_low_fa": "Highest validation danger-clip hit rate; ties prefer fewer safe false alarms, then higher precision.",
    "balanced_window_f1": "Highest validation window F1; ties prefer higher hit rate, then fewer safe false alarms.",
    "precision_guard": "Prefer validation window precision >= 0.50, then maximize hit rate and minimize false alarms.",
    "low_false_alarm": "Prefer validation safe false alarms <= 2/min, then maximize hit rate and precision.",
}


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `compute_threshold_metrics`

Cette cellule definit `compute_threshold_metrics`. Elle prepare une partie du script.

In [ ]:
def compute_threshold_metrics(df, score_col, split_name, threshold, horizon=1.0, persistence_windows=2):
    y_col = f"danger_within_{horizon:.1f}s"
    y_true = df[y_col].astype(int).to_numpy()
    scores = df[score_col].to_numpy()
    y_pred = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    safe_df = df[df["is_danger_clip"] == 0]
    safe_minutes = 0.0
    for _, safe_group in safe_df.groupby("video_id"):
        if len(safe_group):
            safe_minutes += float(safe_group["time_s"].max()) / 60.0
    safe_minutes = max(1e-6, safe_minutes)

    danger_clips = df[df["is_danger_clip"] == 1]["video_id"].unique().tolist()
    clip_hits = 0
    late_or_missed = 0
    early_times = []
    first_alarm_times = []
    for video_id in danger_clips:
        group = df[df["video_id"] == video_id].sort_values("time_s")
        if group.empty:
            continue
        target = group["target_time_s"].replace("", np.nan).astype(float).dropna()
        if target.empty:
            continue
        target_time = float(target.iloc[0])
        alarms = alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=persistence_windows)
        pre_alarms = [float(t) for t in alarms if float(t) <= target_time + 0.5]
        if pre_alarms:
            first_alarm = min(pre_alarms)
            first_alarm_times.append(first_alarm)
            early = target_time - first_alarm
            early_times.append(early)
            if first_alarm <= target_time:
                clip_hits += 1
            else:
                late_or_missed += 1
        else:
            late_or_missed += 1

    safe_episodes = 0
    for _, group in safe_df.groupby("video_id"):
        safe_episodes += len(alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=persistence_windows))

    return {
        "split": split_name,
        "threshold": float(threshold),
        "n_windows": int(len(df)),
        "positive_windows": int(y_true.sum()),
        "window_ap": safe_auc(average_precision_score, y_true, scores),
        "window_roc_auc": safe_auc(roc_auc_score, y_true, scores),
        "window_precision": float(precision),
        "window_recall": float(recall),
        "window_f1": float(f1),
        "danger_clip_count": int(len(danger_clips)),
        "danger_clip_hit_rate": float(clip_hits / max(1, len(danger_clips))),
        "late_or_missed_clips": int(late_or_missed),
        "avg_early_warning_s": float(np.mean(early_times)) if early_times else np.nan,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
        "safe_false_alarm_episodes": int(safe_episodes),
        "safe_false_alarms_per_min": float(safe_episodes / safe_minutes),
    }


## Fonction `threshold_grid`

Cette cellule definit `threshold_grid`. Elle prepare une partie du script.

In [ ]:
def threshold_grid(df, score_col, split_name, horizon=1.0, persistence_windows=2):
    rows = []
    for threshold in [round(x, 2) for x in np.arange(0.02, 1.0, 0.02)]:
        rows.append(compute_threshold_metrics(df, score_col, split_name, threshold, horizon, persistence_windows))
    return pd.DataFrame(rows)


## Fonction `select_policy`

Cette cellule definit `select_policy`. Elle prepare une partie du script.

In [ ]:
def select_policy(sweep, policy):
    data = sweep.copy()
    if policy == "max_hit_low_fa":
        return data.sort_values(
            ["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision", "median_early_warning_s"],
            ascending=[False, True, False, False],
        ).iloc[0]
    if policy == "balanced_window_f1":
        return data.sort_values(
            ["window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"],
            ascending=[False, False, True, False],
        ).iloc[0]
    if policy == "precision_guard":
        eligible = data[data["window_precision"] >= 0.50]
        if eligible.empty:
            eligible = data
        return eligible.sort_values(
            ["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"],
            ascending=[False, True, False],
        ).iloc[0]
    if policy == "low_false_alarm":
        eligible = data[data["safe_false_alarms_per_min"] <= 2.0]
        if eligible.empty:
            eligible = data
        return eligible.sort_values(
            ["danger_clip_hit_rate", "window_precision", "safe_false_alarms_per_min"],
            ascending=[False, False, True],
        ).iloc[0]
    raise ValueError(policy)


## Fonction `load_prediction_files`

Cette cellule definit `load_prediction_files`. Elle prepare une partie du script.

In [ ]:
def load_prediction_files(ensemble_run):
    files = sorted((ensemble_run / "features").glob("ensemble_predictions_seed*.csv"))
    rows = []
    for path in files:
        match = re.search(r"seed(\d+)", path.stem)
        if not match:
            continue
        rows.append((int(match.group(1)), path))
    return rows


## Fonction `measure_score_combination_ms`

Cette cellule definit `measure_score_combination_ms`. Elle prepare une partie du script.

In [ ]:
def measure_score_combination_ms(prediction_files, candidate_members, repeats=200):
    if not prediction_files:
        return {}
    _, path = prediction_files[0]
    df = pd.read_csv(path)
    rows = []
    for variant, members in candidate_members.items():
        cols = [col for col in members if col in df.columns]
        if not cols:
            continue
        values = df[cols].to_numpy(dtype=np.float32)
        start = time.perf_counter()
        for _ in range(repeats):
            if len(cols) == 1:
                _ = values[:, 0]
            else:
                _ = values.mean(axis=1)
        elapsed = time.perf_counter() - start
        rows.append(
            {
                "variant": variant,
                "member_count": len(cols),
                "score_combination_ms_per_window": float(elapsed * 1000.0 / max(1, repeats * len(values))),
            }
        )
    return {row["variant"]: row for row in rows}


## Fonction `model_latency_table`

Cette cellule definit `model_latency_table`. Elle prepare une partie du script.

In [ ]:
def model_latency_table(source_runs, candidate_members):
    frames = []
    for run in source_runs:
        metrics = resolve(run) / "metrics" / "stability_architecture_comparison.csv"
        if metrics.exists():
            frames.append(pd.read_csv(metrics))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df = df[(df["split"] == "test") & (df["horizon_s"] == 1.0)].copy()
    base = (
        df.groupby("base_architecture", as_index=False)
        .agg(
            model_ms_per_window_mean=("inference_ms_per_window", "mean"),
            model_ms_per_window_std=("inference_ms_per_window", "std"),
            model_size_bytes_mean=("model_size_bytes", "mean"),
        )
        .fillna(0.0)
    )
    by_arch = {row["base_architecture"]: row for _, row in base.iterrows()}
    rows = []
    for variant, members in candidate_members.items():
        member_rows = [by_arch[m] for m in members if m in by_arch]
        rows.append(
            {
                "variant": variant,
                "member_models": "+".join(members),
                "member_count": len(member_rows),
                "sequence_model_ms_per_window_est": float(sum(float(row["model_ms_per_window_mean"]) for row in member_rows)),
                "sequence_model_size_mb_est": float(sum(float(row["model_size_bytes_mean"]) for row in member_rows) / (1024 * 1024)),
            }
        )
    return pd.DataFrame(rows)


## Fonction `parse_realtime_baseline`

Cette cellule definit `parse_realtime_baseline`. Elle prepare une partie du script.

In [ ]:
def parse_realtime_baseline(path):
    if not path.exists():
        return {}
    values = {}
    pattern = re.compile(r"- ([^:]+): `?([^`]+)`?")
    for line in path.read_text(encoding="utf-8").splitlines():
        match = pattern.match(line.strip())
        if not match:
            continue
        key = match.group(1).strip().lower().replace(" ", "_").replace("/", "_")
        raw = match.group(2).strip()
        try:
            values[key] = float(raw)
        except ValueError:
            values[key] = raw
    return values


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    ensemble_run = resolve(args.ensemble_run)
    prediction_files = load_prediction_files(ensemble_run)
    write_json(
        run_dir / "config.json",
        {
            "ensemble_run": str(ensemble_run),
            "prediction_files": [str(path) for _, path in prediction_files],
            "candidates": CANDIDATES,
            "operating_policies": OPERATING_POLICIES,
            "horizon_s": args.horizon,
            "persistence_windows": args.persistence_windows,
        },
    )

    all_sweeps = []
    selected_rows = []
    for seed, path in prediction_files:
        df = pd.read_csv(path)
        for variant in CANDIDATES:
            if variant not in df.columns:
                continue
            val = df[df["split"] == "val"].copy()
            test = df[df["split"] == "test"].copy()
            val_sweep = threshold_grid(val, variant, "val", args.horizon, args.persistence_windows)
            val_sweep["repeat_seed"] = seed
            val_sweep["variant"] = variant
            all_sweeps.append(val_sweep)
            for policy in OPERATING_POLICIES:
                selected = select_policy(val_sweep, policy)
                threshold = float(selected["threshold"])
                test_metrics = compute_threshold_metrics(test, variant, "test", threshold, args.horizon, args.persistence_windows)
                selected_rows.append(
                    {
                        "repeat_seed": seed,
                        "variant": variant,
                        "policy": policy,
                        "selected_threshold": threshold,
                        **{f"val_{k}": selected[k] for k in selected.index if k not in {"split", "threshold"}},
                        **{f"test_{k}": v for k, v in test_metrics.items() if k not in {"split", "threshold"}},
                    }
                )

    sweeps = pd.concat(all_sweeps, ignore_index=True)
    selected = pd.DataFrame(selected_rows)
    sweeps.to_csv(run_dir / "metrics" / "validation_threshold_sweeps.csv", index=False)
    selected.to_csv(run_dir / "metrics" / "operating_points_by_seed.csv", index=False)

    summary_rows = []
    metric_cols = [
        "selected_threshold",
        "test_window_ap",
        "test_window_roc_auc",
        "test_window_precision",
        "test_window_recall",
        "test_window_f1",
        "test_danger_clip_hit_rate",
        "test_late_or_missed_clips",
        "test_avg_early_warning_s",
        "test_median_early_warning_s",
        "test_safe_false_alarm_episodes",
        "test_safe_false_alarms_per_min",
    ]
    for (variant, policy), group in selected.groupby(["variant", "policy"]):
        row = {"variant": variant, "policy": policy, "n_repeats": int(group["repeat_seed"].nunique())}
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean())
            row[f"{col}_std"] = float(group[col].std(ddof=0))
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)

    latency = model_latency_table(args.source_stability_runs, CANDIDATES)
    combo = pd.DataFrame(measure_score_combination_ms(prediction_files, CANDIDATES).values())
    if not latency.empty and not combo.empty:
        latency = latency.merge(combo, on=["variant", "member_count"], how="left")
    baseline = parse_realtime_baseline(resolve(args.realtime_baseline))
    if not latency.empty:
        yolo_ms = float(baseline.get("yolo_pose_ms_frame", 0.0))
        attention_ms = float(baseline.get("attention_model_ms_crop", baseline.get("attention_model_ms_crop", 0.0)) or 0.0)
        ppe_ms = float(baseline.get("ppe_model_ms_crop", 0.0) or 0.0)
        if "attention_model_ms_crop" not in baseline and "attention_model_ms_crop" not in baseline:
            attention_ms = float(baseline.get("attention_model_ms_crop", 0.0) or 0.0)
        latency["yolo_pose_ms_per_frame_reference"] = yolo_ms
        latency["attention_ppe_reference_ms_per_frame"] = attention_ms + ppe_ms
        latency["estimated_total_ms_per_frame_reference"] = (
            latency["yolo_pose_ms_per_frame_reference"]
            + latency["sequence_model_ms_per_window_est"]
            + latency.get("score_combination_ms_per_window", 0.0).fillna(0.0)
            + latency["attention_ppe_reference_ms_per_frame"]
        )
        latency["estimated_fps_reference"] = 1000.0 / latency["estimated_total_ms_per_frame_reference"].replace(0, np.nan)

    summary = summary.merge(latency, on="variant", how="left") if not latency.empty else summary
    summary.to_csv(run_dir / "metrics" / "operating_point_summary.csv", index=False)
    if not latency.empty:
        latency.to_csv(run_dir / "metrics" / "latency_size_summary.csv", index=False)

    lines = ["# Final Operating Point And Latency Audit", ""]
    lines.append("Thresholds are selected on each validation split and then applied to the corresponding test split.")
    lines.append("")
    lines.append("## Operating Policies")
    lines.append("")
    for name, description in OPERATING_POLICIES.items():
        lines.append(f"- `{name}`: {description}")
    lines.append("")
    lines.append("## Best Practical Operating Points")
    lines.append("")
    lines.append("| variant | policy | threshold | test AP | hit | FA/min | precision | median early s | est FPS |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|")
    view = summary.copy()
    view["rank_score"] = (
        view["test_danger_clip_hit_rate_mean"] * 2.0
        + view["test_window_ap_mean"]
        + view["test_window_precision_mean"] * 0.5
        - view["test_safe_false_alarms_per_min_mean"] * 0.03
    )
    for _, row in view.sort_values("rank_score", ascending=False).head(18).iterrows():
        est_fps = row.get("estimated_fps_reference", np.nan)
        lines.append(
            f"| {row['variant']} | {row['policy']} | {row['selected_threshold_mean']:.2f} | "
            f"{row['test_window_ap_mean']:.3f} | {row['test_danger_clip_hit_rate_mean']:.3f} | "
            f"{row['test_safe_false_alarms_per_min_mean']:.3f} | {row['test_window_precision_mean']:.3f} | "
            f"{row['test_median_early_warning_s_mean']:.3f} | {est_fps:.1f} |"
        )
    lines.append("")
    lines.append("## Latency Interpretation")
    lines.append("")
    lines.append("- Latency is estimated from measured per-model sequence inference in repeated-split runs plus the existing YOLO/crop reference timing.")
    lines.append("- The ensemble score-combination overhead is measured directly from stored prediction arrays and is negligible relative to YOLO pose extraction.")
    lines.append("- These are research-machine estimates, not a deployment SLA.")
    lines.append("")
    lines.append("## Artifacts")
    lines.append("")
    lines.append(f"- Per-seed operating points: `{run_dir / 'metrics' / 'operating_points_by_seed.csv'}`")
    lines.append(f"- Summary: `{run_dir / 'metrics' / 'operating_point_summary.csv'}`")
    lines.append(f"- Latency/size: `{run_dir / 'metrics' / 'latency_size_summary.csv'}`")
    summary_path = run_dir / "final_operating_point_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Final Operating Point Audit", f"- Summary: `{summary_path}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Audit validation-selected final operating points and latency estimates.")
    parser.add_argument("--run-name", default="exp_033_final_operating_points")
    parser.add_argument("--ensemble-run", default="runs/exp_029_sequence_ensemble_stability")
    parser.add_argument("--source-stability-runs", nargs="+", default=["runs/exp_015_sequence_stability_10seed_shortlist", "runs/exp_021_sequence_stability_extra10"])
    parser.add_argument("--realtime-baseline", default="runs/exp_006_final_research/metrics/realtime_viability.md")
    parser.add_argument("--horizon", type=float, default=1.0)
    parser.add_argument("--persistence-windows", type=int, default=2)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_033_final_operating_points_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_operating_point_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
